<a href="https://colab.research.google.com/github/human530/MD.Piece/blob/claude%2Fautoresearch-feature-ynJbT/autoresearch_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Autoresearch on Colab
基於 [karpathy/autoresearch](https://github.com/karpathy/autoresearch)

**使用方式：** Runtime → Change runtime type → 選 GPU (T4 或更高)

> T4 (Turing) 也可以跑！本 notebook 會自動偵測 GPU 並 patch Flash Attention fallback。

In [ ]:
# 確認 GPU 並偵測架構
!nvidia-smi
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print(f"\n✅ GPU: {name} (compute capability: {cap[0]}.{cap[1]})")
    if cap[0] >= 8:
        print("   Ampere+ → Flash Attention 3 原生支援")
    else:
        print("   Pre-Ampere → 將自動套用 SDPA fallback patch")
else:
    print("❌ 未偵測到 GPU！請到 Runtime → Change runtime type → 選 GPU")

In [12]:
# 安裝 uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ['PATH']

downloading uv 0.10.12 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!


In [13]:
# Clone autoresearch
!git clone https://github.com/karpathy/autoresearch.git
%cd autoresearch

Cloning into 'autoresearch'...
remote: Enumerating objects: 188, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 188 (delta 0), reused 0 (delta 0), pack-reused 186 (from 2)
Receiving objects: 100% (188/188), 529.07 KiB | 12.02 MiB/s, done.
Resolving deltas: 100% (102/102), done.
/content/autoresearch/autoresearch/autoresearch


In [ ]:
# 安裝依賴
!uv sync

# 下載 T4 patch 腳本並套用（Pre-Ampere GPU 自動 patch）
!curl -sL https://raw.githubusercontent.com/human530/MD.Piece/claude/autoresearch-feature-ynJbT/colab_t4_patch.py -o /tmp/t4_patch.py
!python /tmp/t4_patch.py

In [15]:
# 資料準備（下載 + tokenizer，約 2 分鐘）
!uv run prepare.py --num-shards 10

Cache directory: /root/.cache/autoresearch

Data: all 11 shards already downloaded at /root/.cache/autoresearch/data

Tokenizer: already trained at /root/.cache/autoresearch/tokenizer

Done! Ready to train.


In [ ]:
# 跑一次訓練實驗（約 5 分鐘）
!uv run train.py 2>&1 | tee /tmp/run.log

# 提取結果
import re
log = open("/tmp/run.log").read()
bpb_match = re.findall(r"val_bpb[:\s]+([\d.]+)", log)
loss_match = re.findall(r"train_loss[:\s]+([\d.]+)", log)
step_match = re.findall(r"step[:\s]+(\d+)", log)

val_bpb = float(bpb_match[-1]) if bpb_match else None
train_loss = float(loss_match[-1]) if loss_match else None
steps = int(step_match[-1]) if step_match else None

print(f"\n{'='*40}")
print(f"val_bpb:    {val_bpb}")
print(f"train_loss: {train_loss}")
print(f"steps:      {steps}")
print(f"{'='*40}")

## 回傳結果到 MD.Piece API

設定你的 MD.Piece 後端 URL（如果是本機開發，可使用 ngrok 等工具暴露 port 8000）。

In [ ]:
# 回傳實驗結果到 MD.Piece（修改 API_URL 為你的後端位址）
import requests, json
from datetime import datetime

API_URL = "http://localhost:8000"  # @param {type:"string"}
EXPERIMENT_NAME = "baseline-t4"    # @param {type:"string"}
NOTES = "T4 baseline run"          # @param {type:"string"}

payload = {
    "name": EXPERIMENT_NAME,
    "val_bpb": val_bpb,
    "train_loss": train_loss,
    "steps": steps,
    "notes": NOTES,
    "kept": True,
}

try:
    r = requests.post(f"{API_URL}/research/", json=payload, timeout=10)
    if r.ok:
        print(f"✅ 已回傳到 MD.Piece: {r.json()}")
    else:
        print(f"⚠️ API 回傳錯誤 ({r.status_code}): {r.text}")
        print(f"   結果已存在本地，可稍後手動上傳")
except requests.exceptions.ConnectionError:
    print(f"⚠️ 無法連線到 {API_URL}")
    print(f"   請確認後端已啟動，或使用 ngrok 暴露 localhost:8000")
    print(f"\n📋 手動提交指令：")
    print(f'curl -X POST {API_URL}/research/ -H "Content-Type: application/json" -d \'{json.dumps(payload)}\'')